# BM25 full-text and hybrid search — a walkthrough (Python SDK)

How keyword search works in CyborgDB, end to end: **designating** a field, **ingesting**,
**standalone** text queries, and **hybrid** queries that fuse keywords with vector similarity.

This is a teaching notebook, not a benchmark. It runs in seconds on a 12-document corpus you can
read in one screen — which is the point. Every ranking below is one you can check by eye, and the
hybrid section reproduces the engine's fusion arithmetic in plain Python so nothing is a black box.

**What you need:** a running `cyborgdb-service` with BM25 support, and a kernel with the SDK's
runtime dependencies (`numpy`, `pydantic`, `urllib3`, `python-dateutil`, `typing-extensions`,
`python-dotenv`, `requests`). You do **not** need `cyborgdb` itself installed — the setup cell puts
this checkout on `sys.path`, so the notebook always exercises the SDK it ships with rather than
whichever release the kernel happens to have. Point `CYBORGDB_BASE_URL` at the service (default
`http://localhost:8000`) and set `CYBORGDB_API_KEY`. Unlike the in-process `cyborgdb_core` build of this walkthrough, the index
lives on the *service's* storage rather than in this process's memory — so the notebook picks a
fresh index name each run and deletes it at the end.

**The one-paragraph version.** BM25 is turned on by *designating metadata fields* as `full_text`
at index creation — there is no enable flag, and it cannot be added later. Once at least one field
is designated, you get two doors: `query_metadata(text=...)` when you have no query vector, and
`query(query_vectors=..., text=...)` when you do, which fuses the two rankings.

| Query | Door | Ranks by |
|---|---|---|
| Filters only | `query_metadata(filters=...)` | nothing (unordered, or `order_by`) |
| Full text | `query_metadata(text=...)` | BM25 score |
| Hybrid | `query(query_vectors=..., text=...)` | weighted RRF of BM25 + vector rank |

**If you know the `cyborgdb_core` version of this notebook**, four things differ here and are
called out where they bite: the key is bound at `create_index` instead of passed on every call,
`index_config()` is replaced by the `metadata_schema` / `bm25` properties, `query_metadata()`
without `text` returns bare id strings, and `distance` is opt-in via `include=[...]`.

In [1]:
# ---------------------------------------------------------------------------
# SETUP
# ---------------------------------------------------------------------------
import contextlib
import inspect
import logging
import os
import re
import sys
import time
import uuid
from pathlib import Path

import numpy as np

# This notebook ships inside the cyborgdb-py checkout, and its whole point is to exercise
# *that* SDK -- so put the repo ahead of whatever `cyborgdb` the kernel may have installed.
# Set CYBORGDB_REPO if you are running from outside the tree.
def _find_repo(start):
    for candidate in (start, *start.parents):
        if (candidate / "cyborgdb" / "__init__.py").is_file():
            return candidate
    return None

REPO = (Path(os.environ["CYBORGDB_REPO"]).expanduser().resolve()
        if os.getenv("CYBORGDB_REPO") else _find_repo(Path.cwd().resolve()))
assert REPO is not None, (
    "Could not find the cyborgdb-py checkout above "
    f"{Path.cwd()}. Run this notebook from inside the repo, or set CYBORGDB_REPO.")
sys.path.insert(0, str(REPO))

import cyborgdb

# An installed SDK that predates BM25 fails here with a usable message rather than an
# AttributeError several cells down. Note this checks the *client*; a current SDK against an
# old service will instead fail at create_index with a 4xx.
assert "text_fields" in inspect.signature(cyborgdb.Client.create_index).parameters, (
    f"cyborgdb {cyborgdb.__version__} at {cyborgdb.__file__} has no BM25 surface. "
    "Install a version that does, then RESTART THE KERNEL.")

BASE_URL  = os.getenv("CYBORGDB_BASE_URL", "http://localhost:8000")
INDEX_KEY = bytes(range(1, 33))      # 32-byte demo KEK -- generate securely in practice
DIM       = 9                        # tiny on purpose; see the vectors cell
SUFFIX    = uuid.uuid4().hex[:8]     # index names are service-global and persist; keep runs apart


def service_error(exc):
    """The service's own sentence, dug out of the SDK's wrapped HTTP dump.

    Every ApiException is re-raised as a ValueError whose text is the entire HTTP
    response -- headers, body, and the deserialized pydantic model. For a walkthrough
    we want the one line that says why.
    """
    s = str(exc)
    for marker in ('"msg":"Value error, ', '"detail":"'):
        if marker in s:
            s = s.split(marker, 1)[1].split('"', 1)[0]
            break
    else:
        s = s.splitlines()[0]
    return re.sub(r"^Failed to [^:]+: ", "", s)   # drop the SDK's own wrapper prefix


@contextlib.contextmanager
def quiet_sdk_errors():
    """Silence the SDK's pre-raise logging while we provoke errors on purpose.

    Every wrapped ApiException is logged at ERROR with the entire HTTP response
    before being re-raised, which buries the point of a demonstration cell.
    """
    sdk_log = logging.getLogger("cyborgdb")
    previous = sdk_log.level
    sdk_log.setLevel(logging.CRITICAL)
    try:
        yield
    finally:
        sdk_log.setLevel(previous)


client = cyborgdb.Client(BASE_URL, api_key=os.getenv("CYBORGDB_API_KEY", ""))
print("repo   ", REPO)
print("sdk    ", cyborgdb.__version__, "|", cyborgdb.__file__)
assert Path(cyborgdb.__file__).is_relative_to(REPO), (
    "imported an installed cyborgdb, not the one in this checkout")
print("service", client.get_health())

/opt/homebrew/Caskroom/miniconda/base/envs/cyborgdb-core/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
SSL verification is disabled. Not recommended for production.


repo    /Users/cyborg-jim/Documents/repos/cyborgdb-py
sdk     0.17.0rc1.dev5 | /Users/cyborg-jim/Documents/repos/cyborgdb-py/cyborgdb/__init__.py
service {'status': 'healthy', 'api_version': 'v1', 'version': '0.17.1.dev1787320760'}


## The corpus

Twelve short documents across four topics — encryption, indexing, text search, and operations.
Each has a `title` and a `body` (both will be full-text searchable), plus an `author` and a `year`
(ordinary filterable fields).

Two documents are placed deliberately, and the hybrid section turns on them:

- **`ops-1`** is the *keyword trap*. Its title and body both say "search" and "encrypted", so BM25
  loves it — but it is about grepping log archives, so it is nowhere near the query semantically.
- **`crypto-2`** is the opposite. It is squarely on topic but never uses the query's words, so BM25
  cannot see it at all.

A good hybrid ranking should put **`crypto-1`** — which both legs like — above either of them.

In [2]:
# ---------------------------------------------------------------------------
# CORPUS  (id, topic, title, body, author, year)
# ---------------------------------------------------------------------------
DOCS = [
    # -- encryption ---------------------------------------------------------
    ("crypto-1", "crypto", "Encrypted vector search",
     "Searching over encrypted embeddings without ever decrypting them on the server.",
     "ann", 2026),
    ("crypto-2", "crypto", "Key management and rotation",
     "Envelope wrapping, key hierarchies, and how to rotate a tenant key safely.",
     "bob", 2025),
    ("crypto-3", "crypto", "Confidential computing",
     "Running a workload inside an enclave so the host never observes plaintext.",
     "ann", 2026),
    # -- indexing -----------------------------------------------------------
    ("ann-1", "ann", "Tuning IVF probe counts",
     "Choosing n_lists, and how many lists to probe before recall stops improving.",
     "cy", 2025),
    ("ann-2", "ann", "Product quantization",
     "Compressing vectors into short codes so a large index still fits in memory.",
     "bob", 2024),
    ("ann-3", "ann", "Recall and rerank",
     "Rerank a wide candidate window with exact distances to recover lost recall.",
     "cy", 2026),
    # -- text search --------------------------------------------------------
    ("text-1", "text", "How BM25 ranks documents",
     "Term frequency saturates, and long documents are normalized against the average length.",
     "ann", 2025),
    ("text-2", "text", "Stopwords and stemming",
     "An analyzer lowercases, splits, drops stopwords, then stems each remaining token.",
     "cy", 2024),
    ("text-3", "text", "Inverted index basics",
     "A posting list maps one term to the documents that contain it; a search engine walks those lists.",
     "bob", 2026),
    # -- operations ---------------------------------------------------------
    ("ops-1", "ops", "Searching archived logs",
     "Grepping encrypted backup archives when an incident search needs yesterday's logs.",
     "ann", 2024),
    ("ops-2", "ops", "Backup and restore drills",
     "Practice restoring from a snapshot long before you actually need it.",
     "bob", 2025),
    ("ops-3", "ops", "Capacity planning",
     "Estimating disk growth from the ingest rate and the retention window.",
     "cy", 2024),
]

print(f"{'id':10s} {'topic':7s} {'title':32s} {'author':7s} year")
print("-" * 68)
for doc_id, topic, title, body, author, year in DOCS:
    print(f"{doc_id:10s} {topic:7s} {title:32s} {author:7s} {year}")
print(f"\n{len(DOCS)} documents")

id         topic   title                            author  year
--------------------------------------------------------------------
crypto-1   crypto  Encrypted vector search          ann     2026
crypto-2   crypto  Key management and rotation      bob     2025
crypto-3   crypto  Confidential computing           ann     2026
ann-1      ann     Tuning IVF probe counts          cy      2025
ann-2      ann     Product quantization             bob     2024
ann-3      ann     Recall and rerank                cy      2026
text-1     text    How BM25 ranks documents         ann     2025
text-2     text    Stopwords and stemming           cy      2024
text-3     text    Inverted index basics            bob     2026
ops-1      ops     Searching archived logs          ann     2024
ops-2      ops     Backup and restore drills        bob     2025
ops-3      ops     Capacity planning                cy      2024

12 documents


## 1. Designation — turning BM25 on

**A field is searchable because you said so at creation time.** There is no `enable_bm25=True`.
An index with at least one `full_text` field supports text search; an index with none writes no
BM25 config, stages no postings, and pays nothing for the feature.

`text_fields=[...]` is the short way to say it.

In [3]:
# ---------------------------------------------------------------------------
# CREATE  (the short form)
# ---------------------------------------------------------------------------
index = client.create_index(
    f"bm25_walkthrough_{SUFFIX}", INDEX_KEY, dimension=DIM,
    text_fields=["title", "body"],       # <- this is what enables BM25
    metadata_schema={                    # ordinary fields, declared as usual
        "author": {"filterable": True},
        "year":   {"filterable": True},
    },
    bm25_k1=1.2,    # term-frequency saturation (default 1.2)
    bm25_b=0.75,    # length-normalization strength (default 0.75)
)

# The SDK reads config back through two properties rather than a single index_config() dict.
# Both refetch from the service on every read.
print("metadata_schema (normalized -- all three flags always reported):")
for field, flags in sorted(index.metadata_schema.items()):
    print(f"  {field:8s} {flags}")
print("\nbm25:", index.bm25)

metadata_schema (normalized -- all three flags always reported):
  author   {'filterable': True, 'pattern': False, 'full_text': False}
  body     {'filterable': False, 'pattern': False, 'full_text': True}
  title    {'filterable': False, 'pattern': False, 'full_text': True}
  year     {'filterable': True, 'pattern': False, 'full_text': False}

bm25: {'k1': 1.2, 'b': 0.75, 'analyzer_version': 'en/1;sw=2de831ef7be676b6;snowball-english-utf8-3.0.1'}


### Why `title` came back `filterable: False`

Look at the normalized schema above: `title` and `body` are `full_text: True` **and**
`filterable: False`, even though you never wrote that. An analyzed field is not exact-match
indexed — the two are different storage. So `full_text` implies `filterable: False`, and writing
both explicitly is an error rather than a silent preference.

`full_text` + `pattern` is an error too, and always will be: the regex dictionary stores field
values, so putting prose there is junk postings plus a vocabulary-shaped leak.

**If you need a field both searchable and exact-match filterable, store it twice under two names.**

> **SDK gotcha — the long form needs `filterable: False` spelled out.** The wire model defaults
> `filterable` to `true`, and the SDK materializes that default into the request, so a policy
> written as `{"full_text": True}` reaches the service as an *explicit* `filterable: true` and is
> rejected. Through this SDK the long form is `{"full_text": True, "filterable": False}`. The
> `text_fields=[...]` sugar sidesteps the whole issue and is what you should reach for; the cell
> below shows both, and the rejection you get if you forget.

In [4]:
# ---------------------------------------------------------------------------
# The long form, and the combinations that are rejected
# ---------------------------------------------------------------------------
longhand = client.create_index(
    f"bm25_walkthrough_longhand_{SUFFIX}", INDEX_KEY, dimension=DIM,
    metadata_schema={
        # filterable: False is mandatory here, not decorative -- see the note above.
        "title":  {"full_text": True, "filterable": False},
        "body":   {"full_text": True, "filterable": False},
        "author": {"filterable": True},
        "year":   {"filterable": True},
    },
)
print("sugar == longhand:", longhand.metadata_schema == index.metadata_schema)
longhand.delete_index()

# These are errors, not warnings -- the alternative is an index that quietly does not do
# what the schema appears to say.
print()
for label, schema in [
    ("full_text, filterable left default", {"title": {"full_text": True}}),
    ("full_text + filterable",             {"title": {"full_text": True, "filterable": True}}),
    ("full_text + pattern",                {"title": {"full_text": True, "filterable": False,
                                                      "pattern": True}}),
]:
    try:
        with quiet_sdk_errors():
            client.create_index(f"rejected_{uuid.uuid4().hex[:6]}", INDEX_KEY,
                                dimension=DIM, metadata_schema=schema)
        print(f"  {label:36s} -> accepted (unexpected!)")
    except ValueError as e:
        print(f"  {label:36s} -> {service_error(e)}")

sugar == longhand: True

  full_text, filterable left default   -> full_text=true is incompatible with an explicit filterable=true — a full_text field is analyzed rather than exact-match indexed, so it implies filterable=false
  full_text + filterable               -> full_text=true is incompatible with an explicit filterable=true — a full_text field is analyzed rather than exact-match indexed, so it implies filterable=false
  full_text + pattern                  -> pattern=true requires filterable=true (the regex dictionary is only built for filterable fields)


## 2. Ingestion

**Nothing special happens at upsert.** Text is read from the designated fields' metadata values —
you do not pass text separately, and `contents` is never analyzed (it stays arbitrary binary).

The vectors here are hand-built rather than embedded, so the notebook needs no model and no
download. Each topic gets its own axis pair, with a small spread inside the topic. That is enough
for "same topic = nearby" to hold, which is all the hybrid section needs.

Two behaviours worth knowing:

- A `full_text` field that is **missing**, or whose value is not a string, is skipped for that item
  (logged by id only, never value bytes) and excluded from that field's document count.
- An **empty or all-stopword** value is perfectly legal. It simply contributes no terms.

One thing the in-process version does not have to think about: **the service indexes
asynchronously**, so a query fired immediately after an upsert can miss documents. A short sleep
is fine in a notebook; production code polls or tolerates the lag.

In [5]:
# ---------------------------------------------------------------------------
# VECTORS + UPSERT
# ---------------------------------------------------------------------------
# One axis pair per topic: axis `a` marks the topic, axis `a+1` spreads documents within it.
# The last axis carries a tiny per-document value whose only job is to break distance ties, so
# the vector ranking is strictly ordered and the fusion arithmetic later has one right answer.
TOPIC_AXIS    = {"crypto": 0, "ann": 2, "text": 4, "ops": 6}
TIEBREAK_AXIS = 8

vectors, _seen = {}, {}
for g, (doc_id, topic, *_rest) in enumerate(DOCS):
    i = _seen[topic] = _seen.get(topic, -1) + 1
    v = np.zeros(DIM, dtype=np.float32)
    v[TOPIC_AXIS[topic]]     = 1.0             # which topic
    v[TOPIC_AXIS[topic] + 1] = 0.15 * (i + 1)  # where in the topic
    v[TIEBREAK_AXIS]         = 0.001 * g       # unique per document
    vectors[doc_id] = v

# No index_key= here, or on any read below: the SDK bound the key at create_index.
index.upsert(
    [
        {
            "id": doc_id,
            "vector": vectors[doc_id].tolist(),
            "metadata": {"title": title, "body": body, "author": author, "year": year},
        }
        for doc_id, topic, title, body, author, year in DOCS
    ]
)
print(f"upserted {len(DOCS)} documents")

time.sleep(3)   # let the service finish indexing before the first query

# At 12 documents an untrained index brute-force scans, which is exact. On a real corpus you
# would call index.train() once enough vectors have accumulated; BM25 needs no training either
# way -- postings are staged at upsert.
print("is_trained:", index.is_trained())

upserted 12 documents
is_trained: False


## 3. Standalone text query — the no-vector door

`query_metadata(text=...)` runs BM25 over the designated fields and returns rows ranked by score,
highest first, ties broken by id.

**The return type depends on whether you passed `text`.** With `text`, you get
`{"id", "score"}` dicts. Without it — a filter-only query — you get **bare id strings**, because
there is nothing to score and no order to report. That is a hard type switch, not an absent key,
so a helper that formats both has to check.

(The in-process `cyborgdb_core` API instead returns dicts on both paths and simply omits `score`.
If you are porting code between the two, this is the line that breaks.)

In [6]:
# ---------------------------------------------------------------------------
# A FIRST TEXT QUERY
# ---------------------------------------------------------------------------
TITLE = {d[0]: d[2] for d in DOCS}      # id -> title, for readable output

def show(rows, label, n=None):
    print(f"{label}")
    if not rows:
        print("   (no matches)")
        return rows
    for rank, r in enumerate(rows[:n] if n else rows, 1):
        # Scored rows are dicts; a filter-only row is just the id.
        doc_id = r if isinstance(r, str) else r["id"]
        score  = f"{r['score']:.4f}" if isinstance(r, dict) and "score" in r else "-"
        print(f"  {rank:>2}. {doc_id:10s} {score:>8s}  {TITLE[doc_id]}")
    return rows

# -> List[MetadataResult]: {"id", "score"} rows, descending score
hits = index.query_metadata(text="encrypted search")
show(hits, 'query_metadata(text="encrypted search")')

# The filter-only path: same row type, just nothing to score.
# -> List[MetadataResult]: {"id"} rows -- no score, and unordered
rows = index.query_metadata(filters={"author": "ann"})
show(rows, '\nquery_metadata(filters={"author": "ann"})')
print(f"\n   raw rows: {rows}")
print(f"   raw hits: {hits[:1]} ...")

# The MetadataResult contract, checked rather than claimed: `id` on every row,
# `score` only where there is one -- absent, not None. A TypedDict is a plain
# dict at runtime, so this is what "-> List[MetadataResult]" actually buys.
assert all(set(r) == {"id", "score"} for r in hits), hits
assert all(set(r) == {"id"} for r in rows), rows
assert all(isinstance(r, dict) for r in hits + rows)
print("\n   both paths conform to List[MetadataResult]:"
      " text -> {'id','score'}, filter -> {'id'}")

query_metadata(text="encrypted search")
   1. crypto-1     6.7081  Encrypted vector search
   2. ops-1        4.3815  Searching archived logs
   3. text-3       1.1895  Inverted index basics

query_metadata(filters={"author": "ann"})
   1. text-1            -  How BM25 ranks documents
   2. crypto-3          -  Confidential computing
   3. ops-1             -  Searching archived logs
   4. crypto-1          -  Encrypted vector search

   raw rows: [{'id': 'text-1'}, {'id': 'crypto-3'}, {'id': 'ops-1'}, {'id': 'crypto-1'}]
   raw hits: [{'id': 'crypto-1', 'score': 6.708052158355713}] ...

   both paths conform to List[MetadataResult]: text -> {'id','score'}, filter -> {'id'}


### Reading that ranking

Three things decided it, and each is worth seeing on its own.

**The analyzer.** Every value and every query goes through: lowercase → split on non-alphanumeric →
drop stopwords → Porter2 stem. So `Searching`, `search` and `searches` are all the same term, and
`the`, `and`, `of` are not terms at all. This is English-only in v1, and the analyzer's identity is
stamped into the index at creation (you saw it as `analyzer_version` above) — opening the index
with a build whose analyzer differs is a loud error, not silent recall loss.

**Document frequency.** A term that appears in many documents discriminates less and contributes
less. `search` is scattered around this corpus; `enclave` appears once.

**Per-field length normalization.** Statistics are tracked *per field*, so a 3-word title is scored
against the average title length and a 14-word body against the average body length. A short title
is not treated as pathologically short just because bodies are longer.

The next cell demonstrates the analyzer by observation — no expected values, just what comes back.

In [7]:
# ---------------------------------------------------------------------------
# THE ANALYZER, BY OBSERVATION
# ---------------------------------------------------------------------------
def ids(text, **kw):
    # -> List[MetadataResult]: {"id", "score"} rows, descending score; this helper keeps only the ids
    return [r["id"] for r in index.query_metadata(text=text, **kw)]

print("stemming -- these three queries are the same term after Porter2:")
for q in ("search", "searching", "searches"):
    print(f"  {q:12s} -> {ids(q)}")

print("\n...and so are these (both stem to 'encrypt'):")
for q in ("encrypted", "encryption"):
    print(f"  {q:12s} -> {ids(q)}")

print("\ndegenerate input is defined-empty, never an error:")
for label, q in [("all stopwords", "the and of"),
                 ("punctuation only", "--- ... !!!"),
                 ("term not in corpus", "zzzznotaterm"),
                 ("empty-ish", "   ")]:
    print(f"  {label:20s} {q!r:18s} -> {ids(q)}")

print("\ncase and punctuation are irrelevant:")
print("  'ENCRYPTED, Search!' ->", ids("ENCRYPTED, Search!"))
print("  'encrypted search'   ->", ids("encrypted search"))

stemming -- these three queries are the same term after Porter2:
  search       -> ['crypto-1', 'ops-1', 'text-3']
  searching    -> ['crypto-1', 'ops-1', 'text-3']
  searches     -> ['crypto-1', 'ops-1', 'text-3']

...and so are these (both stem to 'encrypt'):
  encrypted    -> ['crypto-1', 'ops-1']
  encryption   -> ['crypto-1', 'ops-1']

degenerate input is defined-empty, never an error:
  all stopwords        'the and of'       -> []
  punctuation only     '--- ... !!!'      -> []
  term not in corpus   'zzzznotaterm'     -> []
  empty-ish            '   '              -> []

case and punctuation are irrelevant:
  'ENCRYPTED, Search!' -> ['crypto-1', 'ops-1', 'text-3']
  'encrypted search'   -> ['crypto-1', 'ops-1', 'text-3']


## Choosing fields, and weighting them

By default a text query searches **every** designated field. `text_fields` narrows it, and
`text_field_weights` reweights the per-field scores before they are summed.

The weights are parallel to the **searched** fields, not to the index's fields — which is why it is
worth naming `text_fields` explicitly whenever you pass weights, so the correspondence is visible
at the call site instead of depending on the schema's ordering.

In [8]:
# ---------------------------------------------------------------------------
# FIELD SELECTION AND WEIGHTS
# ---------------------------------------------------------------------------
Q = "encrypted search"

# -> List[MetadataResult]: {"id", "score"} rows, descending score
show(index.query_metadata(text=Q),
     f'all designated fields          text={Q!r}')
# -> List[MetadataResult]: {"id", "score"} rows, descending score
show(index.query_metadata(text=Q, text_fields=["title"]),
     '\ntitle only')
# -> List[MetadataResult]: {"id", "score"} rows, descending score
show(index.query_metadata(text=Q, text_fields=["body"]),
     '\nbody only')

# Weighting the title 3x. Watch documents whose match is in the title climb.
# -> List[MetadataResult]: {"id", "score"} rows, descending score
show(index.query_metadata(text=Q, text_fields=["title", "body"],
                          text_field_weights=[3.0, 1.0]),
     '\ntitle x3, body x1')

all designated fields          text='encrypted search'
   1. crypto-1     6.7081  Encrypted vector search
   2. ops-1        4.3815  Searching archived logs
   3. text-3       1.1895  Inverted index basics

title only
   1. crypto-1     3.6716  Encrypted vector search
   2. ops-1        1.5895  Searching archived logs

body only
   1. crypto-1     3.0365  Encrypted vector search
   2. ops-1        2.7920  Searching archived logs
   3. text-3       1.1895  Inverted index basics

title x3, body x1
   1. crypto-1    14.0512  Encrypted vector search
   2. ops-1        7.5606  Searching archived logs
   3. text-3       1.1895  Inverted index basics


[{'id': 'crypto-1', 'score': 14.051243782043457},
 {'id': 'ops-1', 'score': 7.560630798339844},
 {'id': 'text-3', 'score': 1.1894774436950684}]

## Any term, or every term

The default is **OR**: a document matching any query term is a candidate, and matching more terms
scores higher. `require_all_terms=True` switches to **AND**, which drops any document missing a
term no matter how well it scores on the others.

Reach for AND when a missing term means the document is simply wrong, not merely weaker.

In [9]:
# ---------------------------------------------------------------------------
# OR (default) vs AND
# ---------------------------------------------------------------------------
# -> List[MetadataResult]: {"id", "score"} rows, descending score
show(index.query_metadata(text="encrypted search"),
     'OR  (default) -- any term')
# -> List[MetadataResult]: {"id", "score"} rows, descending score
show(index.query_metadata(text="encrypted search", require_all_terms=True),
     '\nAND (require_all_terms=True) -- every term')

OR  (default) -- any term
   1. crypto-1     6.7081  Encrypted vector search
   2. ops-1        4.3815  Searching archived logs
   3. text-3       1.1895  Inverted index basics

AND (require_all_terms=True) -- every term
   1. crypto-1     6.7081  Encrypted vector search
   2. ops-1        4.3815  Searching archived logs


[{'id': 'crypto-1', 'score': 6.708052158355713},
 {'id': 'ops-1', 'score': 4.38154411315918}]

## Composing with filters

A filter and a text query compose: **the filter resolves first, then only the survivors are
scored.** It is a pre-filter, not a post-filter, so `top_k` counts documents that passed the filter.

The subtle part is what does *not* change. Document frequency and the per-field averages stay
**corpus-global** — they are never recomputed over the filtered subset. Filtering down to a single
survivor leaves that survivor's score exactly where it was, which is what makes scores comparable
between a filtered and an unfiltered run.

(This is also why multi-tenant deployments should use index-per-tenant rather than a `tenant_id`
filter: co-mingled tenants would each see blended term rarity.)

In [10]:
# ---------------------------------------------------------------------------
# FILTER + TEXT
# ---------------------------------------------------------------------------
# -> List[MetadataResult]: {"id", "score"} rows, descending score
unfiltered = show(index.query_metadata(text="encrypted search"), 'unfiltered')
# -> List[MetadataResult]: {"id", "score"} rows, descending score
filtered = show(index.query_metadata(text="encrypted search", filters={"author": "ann"}),
                '\nfiltered to author=ann (pre-filter: non-survivors are never scored)')

# The invariant: a survivor's score is identical either way.
before = {r["id"]: r["score"] for r in unfiltered}
print("\nscore unchanged by filtering:")
for r in filtered:
    same = abs(r["score"] - before[r["id"]]) < 1e-9
    print(f"  {r['id']:10s} {before[r['id']]:.6f} -> {r['score']:.6f}  {'same' if same else 'CHANGED'}")

# A filter with no survivors is empty, not an error.
# -> List[MetadataResult]: empty list; no survivors is not an error
show(index.query_metadata(text="encrypted search", filters={"author": "nobody"}),
     '\nfiltered to author=nobody')

unfiltered
   1. crypto-1     6.7081  Encrypted vector search
   2. ops-1        4.3815  Searching archived logs
   3. text-3       1.1895  Inverted index basics

filtered to author=ann (pre-filter: non-survivors are never scored)
   1. crypto-1     6.7081  Encrypted vector search
   2. ops-1        4.3815  Searching archived logs

score unchanged by filtering:
  crypto-1   6.708052 -> 6.708052  same
  ops-1      4.381544 -> 4.381544  same

filtered to author=nobody
   (no matches)


[]

## 4. Hybrid — the has-vector door

Setting `text` on `query()` makes the query hybrid. You get one ranking fused from two: the BM25
ranking and the vector ranking.

**A vector is always required, even at `alpha=0`** — it fixes the batch shape. If you have no
vector, you want `query_metadata(text=...)` instead.

Hybrid rows carry `score` (fused relevance, higher is better) and **never** `distance`: a document
matched by text alone has no vector distance to report, so the field is absent rather than
half-populated. In this SDK `distance` is opt-in anyway — a plain vector query returns bare
`{"id"}` rows unless you ask for more with `include=[...]`. The cell below shows that asking for
`distance` on a *hybrid* query still gets you nothing, which is the invariant, not an oversight.

The query below is staged: the query vector sits in the encryption topic (nearest `crypto-2`,
by construction), and the query text is `"encrypted search"`, which `crypto-1` and `ops-1` both
match strongly.

Read the three rankings against each other rather than memorizing them. What to look for:

- **`ops-1` is present at `alpha=0` and gone by `alpha=1`.** Keywords found it; semantics reject it.
- **`crypto-2` is the reverse** — top of the vector leg, invisible to BM25.
- **`crypto-1` should end up at or near the top of the fused ranking** even though it may lead
  neither leg on its own. Agreement across two legs is what RRF rewards.

In [11]:
# ---------------------------------------------------------------------------
# THE TWO LEGS, THEN THE FUSION
# ---------------------------------------------------------------------------
# Query vector: inside the crypto topic but not sitting exactly on any one document, so the
# vector leg has a strict order rather than a tie.
QVEC = np.zeros(DIM, dtype=np.float32)
QVEC[TOPIC_AXIS["crypto"]]     = 1.0
QVEC[TOPIC_AXIS["crypto"] + 1] = 0.28
QTEXT = "encrypted search"
TOP_K = 5

# alpha=0 is the pure-BM25 endpoint and reproduces the standalone ranking exactly;
# alpha=1 is the pure-vector endpoint. An exact endpoint skips the dead leg entirely.
# -> List[Dict[str, Any]]: {"id", "score"} fused rows -- never "distance"
show(index.query(query_vectors=QVEC, text=QTEXT, top_k=TOP_K, alpha=0.0),
     'alpha=0.0  -- pure BM25 (text leg only)')
# -> List[Dict[str, Any]]: {"id", "score"} fused rows -- never "distance"
show(index.query(query_vectors=QVEC, text=QTEXT, top_k=TOP_K, alpha=1.0),
     '\nalpha=1.0  -- pure vector (vector leg only)')
# -> List[Dict[str, Any]]: {"id", "score"} fused rows -- never "distance"
show(index.query(query_vectors=QVEC, text=QTEXT, top_k=TOP_K, alpha=0.5),
     '\nalpha=0.5  -- fused (the default)')

# What `include` can and cannot get you. Every call below returns
# List[Dict[str, Any]]; only the KEY SET differs, which is what keys() prints.
# Vector rows: {"id"} (+ "distance"/"metadata" when included).
# Hybrid rows: {"id", "score"} -- "distance" is refused even if requested.
def keys(rows):
    return sorted(set().union(*(set(r) for r in rows)))

print("\nkeys returned:")
print("  vector, default                       ",
      keys(index.query(query_vectors=QVEC, top_k=3)))
print("  vector, include=['distance']          ",
      keys(index.query(query_vectors=QVEC, top_k=3, include=["distance"])))
print("  hybrid, default                       ",
      keys(index.query(query_vectors=QVEC, text=QTEXT, top_k=3)))
print("  hybrid, include=['distance']          ",
      keys(index.query(query_vectors=QVEC, text=QTEXT, top_k=3, include=["distance"])),
      " <- asked for, still absent")
print("  hybrid, include=['distance','metadata']",
      keys(index.query(query_vectors=QVEC, text=QTEXT, top_k=3,
                       include=["distance", "metadata"])))

alpha=0.0  -- pure BM25 (text leg only)
   1. crypto-1     0.0164  Encrypted vector search
   2. ops-1        0.0161  Searching archived logs
   3. text-3       0.0159  Inverted index basics

alpha=1.0  -- pure vector (vector leg only)
   1. crypto-2     0.0164  Key management and rotation
   2. crypto-1     0.0161  Encrypted vector search
   3. crypto-3     0.0159  Confidential computing
   4. ann-1        0.0156  Tuning IVF probe counts
   5. text-1       0.0154  How BM25 ranks documents

alpha=0.5  -- fused (the default)
   1. crypto-1     0.0163  Encrypted vector search
   2. ops-1        0.0156  Searching archived logs
   3. text-3       0.0150  Inverted index basics
   4. crypto-2     0.0082  Key management and rotation
   5. crypto-3     0.0079  Confidential computing

keys returned:
  vector, default                        ['id']
  vector, include=['distance']           ['distance', 'id']
  hybrid, default                        ['id', 'score']
  hybrid, include=['distance']   

### How the fusion actually works

BM25 relevance and vector distance have no common unit — one is an unbounded score, the other a
metric distance. They cannot be added. So hybrid queries fuse **positions**, not values:

$$\text{score}(d) \;=\; \sum_{\text{leg}} \frac{w_{\text{leg}}}{k + \text{rank}_{\text{leg}}(d)}$$

with `rank` 1-based, $w_{\text{text}} = 1 - \alpha$, $w_{\text{vector}} = \alpha$, and $k$ =
`rrf_k`. A document missing from a leg simply contributes nothing from it.

That is the whole formula. The next cell recomputes it in Python from the two legs and compares
against what the engine returned — if the two columns match, you have understood the ranking
completely.

In [12]:
# ---------------------------------------------------------------------------
# REPRODUCING THE FUSED SCORES BY HAND
# ---------------------------------------------------------------------------
ALPHA, RRF_K = 0.5, 60.0

# Leg 1: the BM25 ranking (the no-vector door).
# -> List[MetadataResult]: {"id", "score"} rows, descending score; reduced here to a rank order
text_leg = [r["id"] for r in index.query_metadata(text=QTEXT)]
# Leg 2: the vector ranking (query() with no text is an ordinary vector search).
# -> List[Dict[str, Any]]: {"id"} rows -- distance only with include=["distance"]; reduced here to a rank order
vec_leg = [r["id"] for r in index.query(query_vectors=QVEC, top_k=len(DOCS))]

print("text leg  :", text_leg)
print("vector leg:", vec_leg)

def rrf(doc_id, alpha=ALPHA, k=RRF_K):
    """Weighted reciprocal-rank fusion, exactly as the engine computes it."""
    s = 0.0
    if doc_id in text_leg:
        s += (1.0 - alpha) / (k + text_leg.index(doc_id) + 1)      # +1: ranks are 1-based
    if doc_id in vec_leg:
        s += alpha / (k + vec_leg.index(doc_id) + 1)
    return s

# -> List[Dict[str, Any]]: {"id", "score"} fused rows -- never "distance"
fused = index.query(query_vectors=QVEC, text=QTEXT, top_k=TOP_K, alpha=ALPHA, rrf_k=RRF_K)

print(f"\n{'id':10s} {'text rk':>8s} {'vec rk':>7s} {'by hand':>10s} {'engine':>10s}  match")
print("-" * 60)
for r in fused:
    d = r["id"]
    t_rk = text_leg.index(d) + 1 if d in text_leg else None
    v_rk = vec_leg.index(d) + 1 if d in vec_leg else None
    mine = rrf(d)
    print(f"{d:10s} {str(t_rk):>8s} {str(v_rk):>7s} {mine:>10.6f} {r['score']:>10.6f}  "
          f"{'yes' if abs(mine - r['score']) < 1e-6 else 'NO'}")

print("\nNote both legs here retrieve the whole corpus, because window_mult * top_k exceeds 12.")
print("On a real corpus each leg stops at that window, and a document past it counts as absent.")

text leg  : ['crypto-1', 'ops-1', 'text-3']
vector leg: ['crypto-2', 'crypto-1', 'crypto-3', 'ann-1', 'text-1', 'ops-1', 'ann-2', 'text-2', 'ops-2', 'ann-3', 'text-3', 'ops-3']

id          text rk  vec rk    by hand     engine  match
------------------------------------------------------------
crypto-1          1       2   0.016261   0.016261  yes
ops-1             2       6   0.015640   0.015640  yes
text-3            3      11   0.014979   0.014979  yes
crypto-2       None       1   0.008197   0.008197  yes
crypto-3       None       3   0.007937   0.007937  yes

Note both legs here retrieve the whole corpus, because window_mult * top_k exceeds 12.
On a real corpus each leg stops at that window, and a document past it counts as absent.


### The three knobs

| Knob | Default | Raise it to | Lower it to |
|---|---|---|---|
| `alpha` | `0.5` | favor vector similarity | favor keyword relevance |
| `rrf_k` | `60` | reward cross-leg agreement further down each list | trust each leg's top hits more |
| `window_mult` | `3` | let documents deep in both legs still win on fusion | retrieve less per leg |

`window_mult` is a **multiple of `top_k`**, not a count: at `window_mult=3, top_k=5` each leg
retrieves 15 candidates before fusing.

Two properties that are not obvious, and that between them explain most "this knob does nothing"
reports:

- **No positive `rrf_k` lets one leg's rank-1 beat two legs' rank-2.** Solving
  `1/(k+1) = 2/(k+2)` gives `k = 0`. If you want one leg to dominate, that is what `alpha` is for —
  `rrf_k` cannot do it. What `rrf_k` controls is how far down the lists agreement keeps mattering.
- **`rrf_k` is only meaningful as deep as the legs actually retrieved.** At `window_mult=1` and
  `top_k=5` each leg is 5 deep, so every `rrf_k` above roughly 5 makes the same decisions. Sweeping
  `rrf_k` at a shallow window looks like a dead knob. **Sweep the two together** — which is what
  the cell below does.

In [13]:
# ---------------------------------------------------------------------------
# SWEEPING rrf_k AT TWO WINDOW DEPTHS
# ---------------------------------------------------------------------------
print(f"fused top-{TOP_K} ids, alpha=0.5\n")
print(f"{'window_mult':>12s} {'leg depth':>10s} {'rrf_k':>7s}  ranking")
print("-" * 78)
for wm in (1, 5):
    for k in (0.5, 5.0, 60.0, 500.0):
        # -> List[Dict[str, Any]]: {"id", "score"} fused rows -- never "distance"
        got = index.query(query_vectors=QVEC, text=QTEXT, top_k=TOP_K, alpha=0.5,
                          rrf_k=k, window_mult=wm)
        print(f"{wm:>12d} {wm * TOP_K:>10d} {k:>7.1f}  {[r['id'] for r in got]}")
    print()

print("Read down each block: at the shallow window the rankings stop responding to rrf_k long")
print("before they do at the deep one. Neither block lets a small rrf_k pull a one-leg document")
print("above a two-leg document that both legs ranked just below it -- that is the 1/(k+1) bound.")

fused top-5 ids, alpha=0.5

 window_mult  leg depth   rrf_k  ranking
------------------------------------------------------------------------------
           1          5     0.5  ['crypto-1', 'crypto-2', 'ops-1', 'text-3', 'crypto-3']
           1          5     5.0  ['crypto-1', 'crypto-2', 'ops-1', 'text-3', 'crypto-3']
           1          5    60.0  ['crypto-1', 'crypto-2', 'ops-1', 'text-3', 'crypto-3']
           1          5   500.0  ['crypto-1', 'crypto-2', 'ops-1', 'text-3', 'crypto-3']

           5         25     0.5  ['crypto-1', 'crypto-2', 'ops-1', 'text-3', 'crypto-3']
           5         25     5.0  ['crypto-1', 'ops-1', 'text-3', 'crypto-2', 'crypto-3']
           5         25    60.0  ['crypto-1', 'ops-1', 'text-3', 'crypto-2', 'crypto-3']
           5         25   500.0  ['crypto-1', 'ops-1', 'text-3', 'crypto-2', 'crypto-3']

Read down each block: at the shallow window the rankings stop responding to rrf_k long
before they do at the deep one. Neither block lets 

## Constraints worth knowing before you design a schema

- **Designations are fixed for the index's lifetime.** No adding `full_text` to a field later in
  v1. Decide at creation.
- **`full_text` + `filterable` is an error** (v1), and **`full_text` + `pattern` is an error
  always**. Need both searchable and exact-match? Store the value twice under two names.
- **Hybrid narrows which filters are accepted.** A `$regex` on a non-`pattern` field, or any
  predicate on an explicitly non-filterable field, works in a pure `query()` but is rejected once
  `text` is set. The BM25 leg has no post-verify stage, so an inexact survivor set would leave it
  silently unfiltered — a wrong answer instead of an error.
- **`order_by` cannot be combined with `text`.** Text results rank by score; there cannot be two
  ordering authorities.
- **Not in v1:** phrase queries, `NOT`/exclusion, nested boolean expressions, non-English analysis,
  and dual-indexing one field as both analyzed and exact-match.
- **Multi-tenant:** index per tenant. Corpus statistics are per index and deliberately global — a
  `tenant_id` filter would give every tenant blended term rarity.

And four that are specific to driving the service through this SDK:

- **The key is bound once**, at `create_index` / `load_index`. There is no per-call `index_key=`.
- **Index names are service-global and persist.** Re-running a notebook that hardcodes a name
  collides with the previous run's leftovers; this one suffixes with a `uuid4`.
- **`distance` is opt-in** via `include=["distance"]`, and unavailable on hybrid rows entirely.
- **Every service error arrives as a `ValueError` carrying the full HTTP dump**, and is logged at
  `ERROR` with that same dump *before* it is raised. Catch `ValueError` and dig the message out (as
  `service_error` does above) rather than matching on the string — and if you are provoking errors
  on purpose, quiet the `cyborgdb` logger first, which is what `quiet_sdk_errors` is for.

The cell below triggers the errors you are most likely to hit, so you can recognize the messages.

In [14]:
# ---------------------------------------------------------------------------
# THE ERRORS YOU WILL ACTUALLY HIT
# ---------------------------------------------------------------------------
def expect_error(label, fn):
    try:
        with quiet_sdk_errors():
            fn()
        print(f"  {label:38s} -> no error (unexpected!)")
    except ValueError as e:
        msg = service_error(e)
        print(f"  {label:38s} -> {msg[:96]}{'...' if len(msg) > 96 else ''}")

# Every call in this block RAISES ValueError instead of returning rows.
expect_error("text_fields names a filterable field",
             lambda: index.query_metadata(text="search", text_fields=["author"]))
expect_error("text_fields names an unknown field",
             lambda: index.query_metadata(text="search", text_fields=["nosuchfield"]))
expect_error("weights not parallel to text_fields",
             lambda: index.query_metadata(text="search", text_fields=["title", "body"],
                                          text_field_weights=[1.0]))
expect_error("order_by combined with text",
             lambda: index.query_metadata(text="search", order_by="year"))
expect_error("text knob set without text",
             lambda: index.query_metadata(text_fields=["title"]))
expect_error("rrf_k=0 (pass None for the default)",
             lambda: index.query(query_vectors=QVEC, text=QTEXT, rrf_k=0.0))
expect_error("window_mult=0",
             lambda: index.query(query_vectors=QVEC, text=QTEXT, window_mult=0))
expect_error("alpha out of [0, 1]",
             lambda: index.query(query_vectors=QVEC, text=QTEXT, alpha=1.5))
expect_error("hybrid knob set without text",
             lambda: index.query(query_vectors=QVEC, alpha=0.5))

# Pass None -- not 0 -- when you want a default.
# -> List[Dict[str, Any]]: {"id", "score"} fused rows -- never "distance"
ok = index.query(query_vectors=QVEC, text=QTEXT, top_k=TOP_K,
                 alpha=None, rrf_k=None, window_mult=None)
print(f"\n  alpha/rrf_k/window_mult = None -> defaults applied, {len(ok)} results")

# And a text query against an index with no designated field:
plain = client.create_index(f"no_bm25_here_{SUFFIX}", INDEX_KEY, dimension=DIM)
plain.upsert([{"id": "x", "vector": np.zeros(DIM, dtype=np.float32).tolist()}])
time.sleep(2)
print(f"  plain.bm25 = {plain.bm25}\n")
# Also raises: no full_text field means no text leg to run.
expect_error("text query on an index with no full_text",
             lambda: plain.query_metadata(text="search"))
plain.delete_index()

  text_fields names a filterable field   -> Invalid input: text query names 'author', which is not a full_text field on this index; full_tex...
  text_fields names an unknown field     -> Invalid input: text query names 'nosuchfield', which is not a full_text field on this index; ful...
  weights not parallel to text_fields    -> Invalid input: text_field_weights has 1 entries but the query searches 2 field(s); weights must ...
  order_by combined with text            -> Invalid input: order_by is not supported together with query text: full-text results are ranked ...
  text knob set without text             -> Invalid input: text_fields / text_field_weights / require_all_terms require non-empty query text...
  rrf_k=0 (pass None for the default)    -> Invalid input: rrf_k must be > 0 (pass None for the default of 60).
  window_mult=0                          -> Invalid input: window_mult must be >= 1 — it is a MULTIPLE of top_k, not an absolute candidate c...
  alpha out of [0, 1]   

## Recap

1. **Designate at creation.** `text_fields=["title", "body"]`, or
   `{"full_text": True, "filterable": False}` in `metadata_schema`. It cannot be added later, and
   it makes the field non-filterable.
2. **Ingest normally.** Text comes from the designated fields' metadata values. Allow for the
   service's indexing lag before the first query.
3. **`query_metadata(text=...)`** is the no-vector door — BM25 ranking, `{"id", "score"}` dicts
   (bare id strings when you omit `text`). Narrow with `text_fields`, reweight with
   `text_field_weights`, tighten with `require_all_terms`, and compose with `filters`
   (pre-filter; global statistics).
4. **`query(query_vectors=..., text=...)`** is the has-vector door — RRF over both legs, tuned with
   `alpha` (which leg matters), `rrf_k` (how deep agreement counts) and `window_mult` (how deep each
   leg looks). Sweep the last two together. Rows carry `score`, never `distance`.

In [15]:
# ---------------------------------------------------------------------------
# CLEANUP
# ---------------------------------------------------------------------------
index.delete_index()
print("index deleted")
print("indexes still on the service from this run:",
      [n for n in client.list_indexes() if SUFFIX in n])

index deleted
indexes still on the service from this run: []
